# STAT 248 — Method 1: time-lagged team-game regression

**Question (proposal).** After controlling for home court, opponent rest structure, lagged outcomes, and season, do teams perform worse when they are on short rest?

**Model.** For outcome $y \in \{\textit{point\_diff},\ \textit{EFG\_PCT},\ \textit{TOV}\}$, within each $\texttt{TEAM\_ID} \times \texttt{SEASON\_ID}$ series:

$$
y_t = \alpha + \rho\, y_{t-1}
      + \beta_1\, \texttt{is\_back\_to\_back}_t
      + \beta_2\, \texttt{is\_short\_rest}_t
      + \cdots\ \text{(opponent fatigue + home + season dummies)}
      + \varepsilon_t
$$

`is_short_rest` flags **0–1 calendar days rest** (`days_rest $\le 1$`, excluding the season opener). `is_back_to_back` is `days_rest == 0`. Coding is **nested**: a B2B game has **both** short-rest and back-to-back dummies switched on.

**Inference.** Sandwich covariance **clustered by $\texttt{TEAM\_ID}$** across all seasons.

**Sample.** Drop each team-season **opener**: no usable $y_{t-1}$, and $\texttt{days\_rest}$ is undefined prior to league play.

In [ ]:
from pathlib import Path

import sys

import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
SCRIPTS = REPO_ROOT / "scripts"
if not (SCRIPTS / "stat248_lagged_regression.py").exists():
    raise RuntimeError("Run with working directory set to the nba_api repo root")

sys.path.insert(0, str(SCRIPTS))
from panel_structure import sort_panel

from stat248_lagged_regression import (
    fit_lagged_team_models,
    prepare_method1_sample,
    stack_regression_tables,
)

CSV_PATH = REPO_ROOT / "data" / "nba_team_game_panel_stat248.csv"
raw = pd.read_csv(CSV_PATH, parse_dates=["GAME_DATE"])
panel = sort_panel(raw)

OUTCOMES = ("point_diff", "EFG_PCT", "TOV")
method_df = prepare_method1_sample(panel, outcomes=OUTCOMES)
fits = fit_lagged_team_models(method_df, outcomes=OUTCOMES)

len(panel), len(method_df)

### Fit summaries

In [ ]:
for y, fit in fits.items():
    print("\n" + "=" * 80)
    print(y, "n_obs=", int(fit.nobs), "rsquared=", round(fit.rsquared, 4))
    print(fit.summary().tables[1])

### Coefficients side-by-side (cluster SE / $p$-values)

In [ ]:
stack_regression_tables(fits)

### Residual diagnostics (primary outcome: scoring margin)

**Note.** With $y_{t-1}$ in the model we expect **Durbin-Watson to be mechanically pulled toward two** compared with a naive static regression; pooled residual QQ is still informative for glaring outliers/normality.

In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson

fd = fits["point_diff"]
diag = pd.DataFrame(
    {"fitted": fd.fittedvalues, "residual": fd.resid, "cluster": method_df["TEAM_ID"]}
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(diag["fitted"], diag["residual"], alpha=0.09, edgecolors="none", s=20)
axes[0].set_xlabel("Fitted point differential")
axes[0].set_ylabel("OLS residual")
axes[0].axhline(0, color="grey", linestyle="--", lw=1)

rr = diag["residual"]
axes[1].hist(rr, bins=40, density=True, color="#4f6fa9", alpha=0.85)
axes[1].set_xlabel("Residual")
axes[1].set_title("Residual histogram (pooled)")
plt.tight_layout()
plt.show()

print(f"Durbin-Watson (pooled residuals, point_diff): {durbin_watson(rr):.3f}")
bp_lm, bp_pval, _, _ = het_breuschpagan(rr, fd.model.exog)
print(f"Breusch-Pagan LM statistic: {bp_lm:.4f}; p-value: {bp_pval:.4g}")


### 结果怎么写报告（对上表系数）

依据当前样本与簇稳健标准误，可把三个结局分开讲清楚：

| 变量 | **point_diff** | **EFG_PCT** | **TOV** |
|------|-----------------|-------------|---------|
| 自身 **背靠背 β** (`is_back_to_back`) | 显著为负 ≈ −2.8 pt 量级 | **eFG** 显著下降 ~0.8 pp | **不显著**，幅度接近 0 |
| 额外 **短休** (`is_short_rest`，在已为“单日”情形下相对再休一天) | 一般不显著（条件于 B2B 哑变量） | 一般不显著 | 一般不显著 |
| **对手背靠背** | 显著为正（对手更累己方 margin 变好） | 对手 eFG 上升（己方防守变差？需结合语境解释为平均效应） | 边际显著为负 (~10%水平) |

这支持 proposal 的核心叙事：**疲劳最直观体现在得分差与投射效率**；失误通道在这层「球队–单场 + 一季滞后」设定下信号较弱。**Breusch-Pagan / DW** 仅作 pooled 粗检——若统计显示异方差或残余序列相关仍然明显，用后续 **ARIMAX + 滚动 CV** 接住。


In [ ]:
_RESULTS = REPO_ROOT / "results"
_RESULTS.mkdir(parents=True, exist_ok=True)
stack_path = _RESULTS / "method1_lagged_coef_table.csv"
stack_regression_tables(fits).sort_index().to_csv(stack_path)
stack_path.resolve()